Load data and packages

In [1]:
setwd('/home/ethan-xiao/food-allergy-biomarkers/data') #Initial version was within the data folder so could pull directly, but now we're just going to add this
getwd()

[1] "/home/ethan-xiao/food-allergy-biomarkers/data"

In [2]:
library(minfi)
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

In [11]:
#load toptables
tt_114134 <- readRDS("tt_114134.rds")
tt_189148 <- readRDS("tt_189148.rds")

head(tt_114134)
head(tt_189148)

ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cg18478105,-0.130885933,-3.2471360,-1.44565175,0.1532921,0.7060781,-4.744851
cg14361672,-0.061896414,3.9532282,-1.20625112,0.2322863,0.7689351,-5.020572
cg01763666,-0.069383007,1.0322597,-0.83531861,0.4067338,0.8575220,-5.353512
cg12950382,-0.003554835,2.9836735,-0.03998494,0.9682331,0.9954335,-5.663558
cg02115394,0.080290367,-0.2988525,0.55645346,0.5798963,0.9163734,-5.525891
cg13417420,0.028517954,-1.3649961,0.12244867,0.9029383,0.9852026,-5.657556


,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cg18478105,-0.06721685,-5.9136529,-1.3764845,0.17567079,0.9999969,-4.486273
cg14361672,0.06391425,3.9875520,1.0012660,0.32220008,0.9999969,-4.743648
cg01763666,0.23851310,0.4481118,1.0891423,0.28204984,0.9999969,-4.690064
cg12950382,0.03617313,1.3539645,0.2309839,0.81840333,0.9999969,-5.025554
cg02115394,-0.07893715,-3.4339939,-0.5272896,0.60065437,0.9999969,-4.958050
cg13417420,-0.53435575,-4.6566380,-1.7128638,0.09381187,0.9999969,-4.195340


In [12]:
#the isg full panel
isg_panel <- c("MX1", "IFI44L", "PARP9", "IFI27", "IFITM1", "IFIT1", "IFIT3",
               "STAT1", "IRF7", "ISG20", "OAS2", "OAS3", "PSMB8", "EPSTI1")

results_list <- list()

for (gene in isg_panel) {
  #\\b = word boundary, keeps "IFIT1" from also matching "IFIT1B" or similar
  probes <- rownames(ann)[grepl(paste0("\\b", gene, "\\b"), ann$UCSC_RefGene_Name)]
  
  if (length(probes) == 0) {
    cat(gene, "- no probes found, skipping\n")
    next
  }
  
#only keep probes that actually made it into both toptables (some might've gotten dropped in QC/filtering earlier)
  probes <- probes[probes %in% rownames(tt_114134) & probes %in% rownames(tt_189148)]
  
  if (length(probes) == 0) {
    cat(gene, "- probes exist but none survived QC in both cohorts, skipping\n")
    next
  }
  
  for (p in probes) {
    results_list[[paste(gene, p)]] <- data.frame(
      gene = gene,
      probe = p,
      logFC_114134 = tt_114134[p, "logFC"],
      pval_114134  = tt_114134[p, "P.Value"],
      logFC_189148 = tt_189148[p, "logFC"],
      pval_189148  = tt_189148[p, "P.Value"]
    )
  }
}

#And...results
isg_panel_results <- do.call(rbind, results_list)
rownames(isg_panel_results) <- NULL
isg_panel_results

gene,probe,logFC_114134,pval_114134,logFC_189148,pval_189148
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
MX1,cg03215005,0.039224590,0.628383314,-0.003214755,0.94046270
MX1,cg16733866,0.065207771,0.422964657,0.175473063,0.34633765
MX1,cg21930140,-0.011790512,0.852279817,-0.027570893,0.57887257
MX1,cg09337082,-0.012936888,0.888651189,-0.025202562,0.63105871
MX1,cg16817229,-0.122293587,0.144853086,0.016358335,0.76253926
MX1,cg21135483,0.206165717,0.002869909,0.046412822,0.37106446
MX1,cg08244262,-0.126081945,0.141129365,-0.033928563,0.45476117
MX1,cg00397324,-0.233608442,0.108263005,-0.148568581,0.52258620
MX1,cg02203106,-0.097870830,0.282437640,0.049984345,0.34799459


In [13]:
#flag which probes clear p<0.05 bar in each cohort
isg_panel_results$sig_114134 <- isg_panel_results$pval_114134 < 0.05
isg_panel_results$sig_189148 <- isg_panel_results$pval_189148 < 0.05

#how many probes, how many nominally significant, which direction
gene_summary <- aggregate(
  cbind(sig_114134, sig_189148, logFC_114134, logFC_189148) ~ gene,
  data = isg_panel_results,
  FUN = function(x) x 
)

#build manually
library(dplyr)
gene_summary <- isg_panel_results %>%
  group_by(gene) %>%
  summarise(
    n_probes = n(),
    n_sig_114134 = sum(sig_114134),
    n_sig_189148 = sum(sig_189148),
    n_sig_both = sum(sig_114134 & sig_189148),
    n_same_direction = sum(sign(logFC_114134) == sign(logFC_189148)),
    n_hypometh_both = sum(logFC_114134 < 0 & logFC_189148 < 0)
  )

gene_summary


Attaching package: ‘dplyr’


The following object is masked from ‘package:minfi’:

    combine


The following objects are masked from ‘package:Biostrings’:

    collapse, intersect, setdiff, setequal, union


The following object is masked from ‘package:XVector’:

    slice


The following object is masked from ‘package:Biobase’:

    combine


The following object is masked from ‘package:matrixStats’:

    count


The following objects are masked from ‘package:GenomicRanges’:

    intersect, setdiff, union


The following object is masked from ‘package:Seqinfo’:

    intersect


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, setequal, union


The following object is masked from ‘package:generics’:

    explai

gene,n_probes,n_sig_114134,n_sig_189148,n_sig_both,n_same_direction,n_hypometh_both
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
EPSTI1,28,4,0,0,17,7
IFI27,18,0,0,0,9,2
IFI44L,14,2,3,0,7,1
IFIT1,6,3,1,1,5,4
IFIT3,13,0,0,0,9,4
IFITM1,19,4,1,0,8,3
IRF7,28,2,0,0,16,6
ISG20,26,1,0,0,19,9
MX1,52,6,2,0,32,10


In [14]:
colSums(gene_summary[, -1])  # everything except the gene name column

n_probes     n_sig_114134     n_sig_189148       n_sig_both 
             380               41               13                1 
n_same_direction  n_hypometh_both 
             219               89

In [17]:
#load saved bumphunter output
bumps_114134 <- readRDS("bumphunter_114134.rds")
bumps_189148 <- readRDS("bumphunter_189148_B250.rds")

#check structure first
str(bumps_114134, max.level = 1)

head(bumps_114134$table)

List of 6
 $ table          :'data.frame':	38638 obs. of  14 variables:
 $ coef           : num [1:751107, 1] -0.018142 0.050429 -0.042942 -0.000141 0.166477 ...
  ..- attr(*, "dimnames")=List of 2
 $ fitted         : num [1:751107, 1] -0.018142 0.050429 -0.042942 -0.000141 0.166477 ...
  ..- attr(*, "dimnames")=List of 2
 $ pvaluesMarginal: Named num [1:751107] 0.6863 0.3333 0.5882 1 0.0196 ...
  ..- attr(*, "names")= chr [1:751107] "cg26928153" "cg16269199" "cg13869341" "cg24669183" ...
 $ null           :List of 2
 $ algorithm      :List of 12
 - attr(*, "class")= chr "bumps"


,chr,start,end,value,area,cluster,indexStart,indexEnd,L,clusterL,p.value,fwer,p.valueArea,fwerArea
,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
12634,chr5,135414858,135416613,0.4105861,7.390550,299571,590576,590593,18,18,2.979933e-06,0.08,1.415468e-05,0.36
3412,chr12,739953,740338,0.6859428,3.429714,77647,155276,155280,5,5,1.638963e-05,0.34,1.847559e-04,0.98
8249,chr2,30669597,30669863,0.7692717,3.077087,193716,395539,395542,4,14,2.011455e-05,0.40,2.488244e-04,1.00
37973,chr8,144660395,144660772,-0.5151121,3.605785,367437,725909,725915,7,11,3.799415e-05,0.70,1.601714e-04,0.98
23639,chr15,22833172,22833400,-0.4498230,3.148761,120050,239453,239459,7,12,7.077341e-05,0.88,2.324348e-04,1.00
30763,chr22,48314374,48314866,-0.7588537,1.517707,243377,488025,488026,2,3,7.822324e-05,0.88,1.653118e-03,1.00


In [18]:
#get genomic coordinates for every isg panel probe 
probe_coords <- ann[isg_panel_results$probe, c("chr", "pos")]
probe_coords$gene <- isg_panel_results$gene
probe_coords$probe <- isg_panel_results$probe

head(probe_coords)

DataFrame with 6 rows and 4 columns
                   chr       pos        gene       probe
           <character> <integer> <character> <character>
cg03215005       chr21  42792304         MX1  cg03215005
cg16733866       chr21  42792609         MX1  cg16733866
cg21930140       chr21  42796537         MX1  cg21930140
cg09337082       chr21  42792302         MX1  cg09337082
cg16817229       chr21  42798199         MX1  cg16817229
cg21135483       chr21  42830710         MX1  cg21135483

In [19]:
#for each panel probe, check if it falls inside any bump in either cohort's table
check_overlap <- function(probe_row, bump_table) {
  hits <- bump_table[
    bump_table$chr == probe_row$chr &
    bump_table$start <= probe_row$pos &
    bump_table$end >= probe_row$pos,
  ]
  if (nrow(hits) == 0) return(NULL)
  hits$gene <- probe_row$gene
  hits$probe <- probe_row$probe
  hits
}

#run it across every probe for both cohorts
overlaps_114134 <- do.call(rbind, lapply(1:nrow(probe_coords), function(i) {
  check_overlap(probe_coords[i, ], bumps_114134$table)
}))

overlaps_189148 <- do.call(rbind, lapply(1:nrow(probe_coords), function(i) {
  check_overlap(probe_coords[i, ], bumps_189148$table)
}))

overlaps_114134
overlaps_189148

,chr,start,end,value,area,cluster,indexStart,indexEnd,L,clusterL,p.value,fwer,p.valueArea,fwerArea,gene,probe
,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
9796,chr21,42830710,42830710,0.2061657,0.2061657,233968,469632,469632,1,2,0.26646860,1,0.29038107,1,MX1,cg21135483
30298,chr21,42798594,42798594,-0.2336084,0.2336084,233960,469609,469609,1,14,0.14479421,1,0.17743565,1,MX1,cg00397324
30297,chr21,42798131,42798131,-0.2510362,0.2510362,233960,469606,469606,1,14,0.10064501,1,0.13688397,1,MX1,cg15925792
9794,chr21,42795929,42795929,0.1814315,0.1814315,233958,469595,469595,1,1,0.47349796,1,0.48575741,1,MX1,cg13155430
9795,chr21,42799141,42799141,0.2378894,0.2378894,233960,469611,469611,1,14,0.13212800,1,0.16575431,1,MX1,cg21549285
754,chr1,79088559,79088559,0.2509947,0.2509947,16642,34226,34226,1,2,0.10072919,1,0.13696145,1,IFI44L,cg13452062
31580,chr3,122283635,122283635,-0.1790204,0.1790204,257897,514662,514662,1,16,0.50157862,1,0.51249486,1,PARP9,cg27291138
19796,chr11,313120,313120,-0.2392781,0.2392781,56730,111204,111204,1,16,0.12830028,1,0.16220819,1,IFITM1,cg04582010
19797,chr11,314044,314044,-0.1880391,0.1880391,56730,111211,111211,1,16,0.40541245,1,0.42107424,1,IFITM1,cg21625464


,chr,start,end,value,area,cluster,indexStart,indexEnd,L,clusterL,p.value,fwer,p.valueArea,fwerArea,gene,probe
,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
29462,chr21,42792609,42792609,0.1754731,0.1754731,233956,469592,469592,1,12,0.71182583,1,0.72034847,1,MX1,cg16733866
29463,chr21,42795929,42795929,0.4935775,0.4935775,233958,469595,469595,1,1,0.01226598,1,0.05070582,1,MX1,cg13155430
68978,chr21,42792703,42792703,-0.2001768,0.2001768,233956,469593,469593,1,12,0.50800799,1,0.53191720,1,MX1,cg13507964
29467,chr21,42804101,42804101,0.2042305,0.2042305,233963,469621,469621,1,3,0.48038949,1,0.50690360,1,MX1,cg07495530
29461,chr21,42791937,42791937,0.2337949,0.2337949,233956,469583,469583,1,12,0.31912094,1,0.36272382,1,MX1,cg13755924
29465,chr21,42797953,42797953,0.2142746,0.2142746,233960,469605,469605,1,14,0.41813988,1,0.45087675,1,MX1,cg12359279
29466,chr21,42799141,42799141,0.2375022,0.2375022,233960,469611,469611,1,14,0.30310226,1,0.34856409,1,MX1,cg21549285
29464,chr21,42797847,42797899,0.2155511,0.4311023,233960,469601,469602,2,14,0.04868939,1,0.07417550,1,MX1,cg17986793
294641,chr21,42797847,42797899,0.2155511,0.4311023,233960,469601,469602,2,14,0.04868939,1,0.07417550,1,MX1,cg26312951


In [20]:
#just the low nominal p-value bumps in each cohort
low_p_114134 <- overlaps_114134[overlaps_114134$p.value < 0.05, c("gene", "probe", "chr", "start", "end", "value", "p.value")]
low_p_189148 <- overlaps_189148[overlaps_189148$p.value < 0.05, c("gene", "probe", "chr", "start", "end", "value", "p.value")]

low_p_114134
low_p_189148

#which genes show up in both low-p lists
intersect(low_p_114134$gene, low_p_189148$gene)

,gene,probe,chr,start,end,value,p.value
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
19332,IFIT1,cg26974214,chr10,91151885,91152280,-0.1849510,0.00957527
193321,IFIT1,cg16395953,chr10,91151885,91152280,-0.1849510,0.00957527
193322,IFIT1,cg11748577,chr10,91151885,91152280,-0.1849510,0.00957527
22540,EPSTI1,cg12138678,chr13,43566618,43566633,-0.1905643,0.03051675
225401,EPSTI1,cg16327891,chr13,43566618,43566633,-0.1905643,0.03051675


,gene,probe,chr,start,end,value,p.value
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
29463,MX1,cg13155430,chr21,42795929,42795929,0.4935775,0.01226598
29464,MX1,cg17986793,chr21,42797847,42797899,0.2155511,0.04868939
294641,MX1,cg26312951,chr21,42797847,42797899,0.2155511,0.04868939
2010,IFI44L,cg13452062,chr1,79088559,79088769,0.4363814,0.00280460
2009,IFI44L,cg03607951,chr1,79085586,79085713,0.2175158,0.04752503
20101,IFI44L,cg05696877,chr1,79088559,79088769,0.4363814,0.00280460
20091,IFI44L,cg17980508,chr1,79085586,79085713,0.2175158,0.04752503
7016,IFITM1,cg11694510,chr11,313120,313354,0.1790107,0.01639927
70161,IFITM1,cg04582010,chr11,313120,313354,0.1790107,0.01639927


character(0)

In [22]:
saveRDS(isg_panel_results, "isg_panel_results.rds") #Saving results
saveRDS(list(overlaps_114134 = overlaps_114134, overlaps_189148 = overlaps_189148),
        "isg_panel_bumphunter_overlaps.rds")

 Specificity control: ran the same differential methylation pipeline on a 14-gene panel of unrelated interferon-response genes (MX1, IFI44L, PARP9, IFI27, IFITM1, IFIT1, IFIT3, STAT1, IRF7, ISG20, OAS2, OAS3, PSMB8, EPSTI1 — 380 probes total), to test whether ISG15's cross-cohort concordance is a generic pipeline artifact or specific to this locus.
- 13 of 14 genes showed no probe significant in both cohorts. One exception: IFIT1 has one probe (of 6) nominally significant (p<0.05) in both cohorts
- Also checked each panel gene's overlap with each cohort's bumphunter regions individually (not cross-cohort matched)
- Overall, this panel argues against ISG15's signal being a broad, generic signal, with one minor exception (IFIT1)

Precedes the genome-wide concordance scan (notebook 9)